# Análisis de datos – Tesis UBO
## Aluviones y lluvias intensas en la Región Metropolitana: caracterización, modelo de IA y análisis del piloto

**Tesis:** *Entrenamiento de IA para la prevención de desastres naturales* (Pérez y Chabo, 2026).

Este notebook genera los resultados de los capítulos 4 y 5 que se pueden obtener de los datos. Cada sección indica a qué parte de la tesis corresponde:

| Sección del notebook | Sección de la tesis | Fuente |
|---|---|---|
| 1–2. Carga de DesInventar Chile | 3.3 Fuentes de datos | DesInventar Chile (LA RED / UNDRR) |
| 3. Caracterización de eventos en la RM | **4.1** (Objetivo específico 1) | DesInventar Chile |
| 4–5. Datos hidrometeorológicos y unión | **4.1** | CAMELS-CL (Alvarez-Garreton et al., 2018) |
| 6. Entrenamiento y evaluación del modelo de IA | **4.2** (Objetivo específico 2) | DesInventar + CAMELS-CL |
| 7. Selección de casos para los retos | **4.2 / 3.5** | DesInventar + CAMELS-CL |
| 8. Análisis del piloto (pre–post) | **4.3** (Objetivo específico 4) | **Datos propios del piloto** (se completan después de aplicarlo) |
| 9. Exportación a Word | Tablas APA listas para pegar | — |

**Cómo usarlo:** `Entorno de ejecución → Ejecutar todas`. Si una descarga automática falla, la celda te pedirá subir el archivo manualmente (los enlaces están en la sección 1).

> ⚠️ Los resultados del **piloto** (sección 8) **no** salen de estos datasets: salen de los cuestionarios que respondan los participantes. El notebook trae la plantilla y el análisis listo para cuando tengan esos datos.

## 0. Configuración
Puedes ajustar los parámetros de esta celda. Los valores por defecto son los descritos en la metodología de la tesis.

### 📘 Explicación para la defensa — Configuración
**Qué hace:** reúne en un solo lugar todos los parámetros que afectan los resultados: cómo se reconoce la RM, qué tipos de evento se consideran, el período de análisis, la caja geográfica para elegir cuencas y el año de corte entre entrenamiento y prueba. La segunda celda instala librerías y define funciones auxiliares (descargar, pedir un archivo, registrar tablas y figuras).

**Qué entrega:** nada visible; prepara el entorno.

**Por qué así:** centralizar los parámetros hace el análisis **reproducible y auditable**: si el profesor pregunta "¿qué pasa si consideran solo aluviones?" o "¿y si cortan en 2010?", se cambia una línea y se vuelve a ejecutar. Registrar cada tabla con su título y nota permite exportarlas después en formato APA sin copiarlas a mano (menos errores de transcripción).

**Cómo ampliarlo:** convertir los parámetros en un formulario de Colab (`#@param`) o en un archivo `config.json` para correr varios escenarios.

In [ ]:
# ---------------- PARÁMETROS ----------------
REGION_PATRON = r"metropolitana|santiago"   # cómo se reconoce la RM en DesInventar (name1)
REGION_CODIGO = "13"                         # código de región (level1 suele terminar en 13)
ANIO_MIN, ANIO_MAX = 1970, 2014              # período de DesInventar Chile

# Tipos de evento hidrometeorológicos considerados (códigos en inglés y español)
TIPOS_EVENTO = {
    "ALLUVION": "Aluvión", "ALUVION": "Aluvión", "ALUVIÓN": "Aluvión",
    "SPATE": "Avenida torrencial", "AVENIDA": "Avenida torrencial",
    "FLOOD": "Inundación", "INUNDACION": "Inundación", "INUNDACIÓN": "Inundación",
    "RAIN": "Lluvias", "LLUVIAS": "Lluvias", "LLUVIA": "Lluvias",
    "LANDSLIDE": "Deslizamiento", "DESLIZAMIENTO": "Deslizamiento",
    "STORM": "Temporal", "TEMPESTAD": "Temporal", "TEMPORAL": "Temporal",
}

# Caja geográfica aproximada de la RM para elegir cuencas CAMELS-CL
RM_LAT = (-34.30, -32.90)
RM_LON = (-71.75, -69.75)

# Partición temporal del modelo
ANIO_FIN_ENTRENAMIENTO = 2006     # entrenar 1979–2006, evaluar 2007–2014
SEMILLA = 42
DATA_DIR = "datos"
OUT_DIR = "resultados"

In [ ]:
!pip -q install python-docx lxml openpyxl 2>/dev/null
import os, io, re, zipfile, glob, warnings, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import requests
warnings.filterwarnings("ignore")
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.dpi": 120, "font.family": "serif", "font.size": 10})

try:
    from google.colab import files as colab_files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

TABLAS, FIGURAS = [], []   # se van acumulando para exportarlas a Word al final

def registrar_tabla(df, titulo, nota, seccion):
    TABLAS.append({"df": df.copy(), "titulo": titulo, "nota": nota, "seccion": seccion})
    print(f"[Tabla registrada para {seccion}] {titulo}")
    display(df)

def registrar_figura(fig, nombre, titulo, nota, seccion):
    ruta = os.path.join(OUT_DIR, nombre)
    fig.savefig(ruta, dpi=200, bbox_inches="tight")
    FIGURAS.append({"ruta": ruta, "titulo": titulo, "nota": nota, "seccion": seccion})
    plt.show()

def descargar(urls, destino, min_bytes=10_000):
    # Intenta descargar desde una lista de URLs; devuelve la ruta si lo logra.
    if os.path.exists(destino) and os.path.getsize(destino) > min_bytes:
        print("Ya existe:", destino); return destino
    for u in urls:
        try:
            print("Descargando", u, "...")
            r = requests.get(u, timeout=180, headers={"User-Agent": "Mozilla/5.0"})
            if r.status_code == 200 and len(r.content) > min_bytes:
                open(destino, "wb").write(r.content)
                print(f"  OK ({len(r.content)/1e6:.1f} MB)"); return destino
            print("  respuesta", r.status_code, len(r.content), "bytes")
        except Exception as e:
            print("  error:", e)
    return None

def pedir_archivo(mensaje):
    print(mensaje)
    if not EN_COLAB:
        raise FileNotFoundError("Fuera de Colab: copia el archivo en la carpeta 'datos/' y vuelve a ejecutar.")
    subido = colab_files.upload()
    nombre = list(subido.keys())[0]
    ruta = os.path.join(DATA_DIR, nombre)
    open(ruta, "wb").write(subido[nombre])
    return ruta

## 1. Descarga de DesInventar Chile

Enlaces (por si necesitas descargar a mano):
- Página de descargas (buscar **Chile** en *Americas*): https://desinventar.cimafoundation.org/download.html
- Descarga directa Chile: https://desinventar.cimafoundation.org/DesInventar/download_base.jsp?countrycode=chl
- Alternativa: https://www.desinventar.net/DesInventar/download/DI_export_chl.zip
- Consulta en línea (perfil por región): https://www.desinventar.net/DesInventar/profiletab.jsp?countrycode=chl&continue=y

El archivo es un `.zip` que contiene `DI_export_chl.xml` (registros en `<fichas><TR>…</TR></fichas>`). Si lo descargas a mano, súbelo cuando la celda lo pida (sirve el `.zip` o el `.xml`; también un `.csv`/`.xlsx` exportado desde la consulta web).

### 📘 Explicación para la defensa — Descarga de DesInventar
**Qué hace:** intenta descargar automáticamente la base de Chile desde dos servidores oficiales de DesInventar; si ambos fallan, pide subir el archivo a mano.

**Qué entrega:** el archivo `DI_export_chl.zip` en la carpeta `datos/`.

**Por qué DesInventar y no otra fuente:**
- Es la única base **pública, descargable y a escala comunal** que cubre décadas de eventos en Chile con una metodología homogénea (la misma en más de 80 países, usada por UNDRR para el Marco de Sendai).
- **EM-DAT** (base internacional) solo registra desastres grandes (≥10 muertos, ≥100 afectados o declaración de emergencia), por lo que casi no tendría eventos de la RM.
- La base de **Escenarios Hídricos 2030** es más completa (1912–2017), pero no es de descarga libre; queda como fuente de validación.
- Los reportes de **Senapred y Sernageomin** son documentos (PDF), no una base de datos estructurada.

**Limitación a reconocer:** DesInventar llega hasta 2014 y se construye mayoritariamente con prensa, por lo que puede subregistrar eventos pequeños o de comunas con menor cobertura mediática.

In [ ]:
URLS_DI = [
    "https://www.desinventar.net/DesInventar/download/DI_export_chl.zip",
    "https://desinventar.cimafoundation.org/DesInventar/download/DI_export_chl.zip",
    "https://desinventar.cimafoundation.org/DesInventar/download_base.jsp?countrycode=chl",
]
locales = [p for p in glob.glob(os.path.join(DATA_DIR, "*")) if re.search(r"DI_export|desinventar|chl", os.path.basename(p), re.I)]
ruta_di = locales[0] if locales else descargar(URLS_DI, os.path.join(DATA_DIR, "DI_export_chl.zip"))
if ruta_di is None or not (zipfile.is_zipfile(ruta_di) or ruta_di.lower().endswith((".xml", ".csv", ".xlsx", ".xls"))):
    ruta_di = pedir_archivo("No se pudo descargar automáticamente. Sube el ZIP/XML de DesInventar Chile:")
print("Archivo DesInventar:", ruta_di)

## 2. Lectura y limpieza de DesInventar

### 📘 Explicación para la defensa — Lectura y limpieza
**Qué hace:**
1. Lee el XML del ZIP **registro por registro** (`iterparse`), tomando solo los elementos `<TR>` dentro de `<fichas>` (cada uno es una "ficha" = un evento en una comuna).
2. Convierte las columnas numéricas (fallecidos, viviendas, etc.) y normaliza los nombres de evento (DesInventar usa códigos en inglés como `ALLUVION` o en español como `ALUVION`) a seis categorías en español.
3. Filtra la **Región Metropolitana** (por nombre o por código 13), los **tipos hidrometeorológicos** y el período **1970–2014**.
4. Construye la fecha y marca si es **completa** (algunas fichas solo traen año o mes).

**Qué entrega:** la tabla `rm` y el archivo `desinventar_rm_hidrometeorologicos.csv` (datos limpios para revisar en Excel).

**Por qué así:**
- `iterparse` no carga todo el XML en memoria de una vez: es la forma recomendada para XML grandes y evita que Colab se quede sin RAM.
- Se incluyen **seis tipos** (aluvión, avenida, inundación, lluvias, deslizamiento y temporal) y no solo "aluvión" porque en DesInventar Chile hay muy pocas fichas tipificadas como aluvión: muchos aluviones quedaron registrados como "temporal" o "avenida". Usar solo "aluvión" dejaría muy pocos casos para entrenar un modelo.
- Las fichas sin día exacto se conservan para los conteos (sección 3), pero no se usan en el modelo (sección 5), porque ahí hay que saber qué lluvia cayó ese día.

In [ ]:
CAMPOS = ["serial","level0","level1","level2","name0","name1","name2","evento","lugar","fechano","fechames","fechadia",
          "duracion","muertos","heridos","desaparece","damnificados","afectados","vivdest","vivafec","evacuados",
          "reubicados","valorus","valorloc","nhectareas","cabezas","kmvias","latitude","longitude","fuentes"]

def leer_xml_desinventar(fuente):
    from lxml import etree
    filas = []
    parser_kw = dict(events=("end",), tag="TR", recover=True, huge_tree=True)
    ctx = etree.iterparse(fuente, **parser_kw)
    for _, el in ctx:
        padre = el.getparent()
        if padre is not None and padre.tag.lower() == "fichas":
            filas.append({c.tag: (c.text or "").strip() for c in el})
        el.clear()
    return pd.DataFrame(filas)

def leer_desinventar(ruta):
    if zipfile.is_zipfile(ruta):
        z = zipfile.ZipFile(ruta)
        xmls = [n for n in z.namelist() if n.lower().endswith(".xml")]
        if not xmls: raise ValueError(f"El ZIP no contiene XML: {z.namelist()[:10]}")
        with z.open(xmls[0]) as f:
            return leer_xml_desinventar(f)
    if ruta.lower().endswith(".xml"):
        return leer_xml_desinventar(ruta)
    # Exportación manual CSV/Excel desde la consulta web: se normalizan los encabezados
    df = pd.read_excel(ruta) if ruta.lower().endswith(("xlsx","xls")) else pd.read_csv(ruta, sep=None, engine="python", encoding_errors="replace")
    mapa = {"evento":"evento","event":"evento","muertos":"muertos","deaths":"muertos","heridos":"heridos","injured":"heridos",
            "viviendas destruidas":"vivdest","houses destroyed":"vivdest","viviendas afectadas":"vivafec","houses damaged":"vivafec",
            "damnificados":"damnificados","directly affected":"damnificados","afectados":"afectados","indirectly affected":"afectados",
            "evacuados":"evacuados","evacuated":"evacuados","region":"name1","región":"name1","comuna":"name2","municipality":"name2",
            "lugar":"lugar","place":"lugar","fecha":"fecha","date":"fecha","serial":"serial","duracion":"duracion","duración":"duracion"}
    df.columns = [mapa.get(str(c).strip().lower(), str(c).strip().lower()) for c in df.columns]
    if "fecha" in df and "fechano" not in df:
        f = pd.to_datetime(df["fecha"].astype(str).str.replace(r"[^0-9/\-]", "", regex=True), errors="coerce", dayfirst=False)
        df["fechano"], df["fechames"], df["fechadia"] = f.dt.year, f.dt.month, f.dt.day
    return df

di_raw = leer_desinventar(ruta_di)
print("Registros totales leídos:", len(di_raw))
print("Columnas:", list(di_raw.columns)[:40])

di = di_raw.copy()
for c in ["fechano","fechames","fechadia","duracion","muertos","heridos","desaparece","damnificados","afectados",
          "vivdest","vivafec","evacuados","reubicados","valorus","valorloc","kmvias"]:
    if c in di: di[c] = pd.to_numeric(di[c], errors="coerce")
for c in ["name1","name2","level1","evento","lugar"]:
    if c not in di: di[c] = ""
    di[c] = di[c].fillna("").astype(str)
di["evento_norm"] = di["evento"].str.upper().str.strip().map(TIPOS_EVENTO)

es_rm = di["name1"].str.contains(REGION_PATRON, case=False, regex=True) | di["level1"].str.replace(r"\D", "", regex=True).str.endswith(REGION_CODIGO)
print("\nRegistros en la RM (todos los tipos):", es_rm.sum())
print("Valores de name1 detectados como RM:", di.loc[es_rm, "name1"].value_counts().head(5).to_dict())

rm = di[es_rm & di["evento_norm"].notna() & di["fechano"].between(ANIO_MIN, ANIO_MAX)].copy()
rm["fecha"] = pd.to_datetime(dict(year=rm["fechano"], month=rm["fechames"].clip(1, 12).fillna(1),
                                  day=rm["fechadia"].clip(1, 31).fillna(1)), errors="coerce")
rm["fecha_valida"] = rm["fechadia"].between(1, 31) & rm["fechames"].between(1, 12) & rm["fecha"].notna()
print("Registros hidrometeorológicos en la RM:", len(rm), "| con fecha completa:", rm["fecha_valida"].sum())
rm.to_csv(os.path.join(OUT_DIR, "desinventar_rm_hidrometeorologicos.csv"), index=False)
rm.head()

## 3. Caracterización de los eventos en la RM → Tesis 4.1 (Objetivo específico 1)

### 📘 Explicación para la defensa — Caracterización (Objetivo específico 1)
**Qué hace:** estadística descriptiva de los eventos de la RM: número de fichas e impactos por tipo, distribución por mes y por década, y comunas más afectadas.

**Qué entrega:** 2 tablas y 1 figura para la sección 4.1, más el porcentaje de eventos entre mayo y agosto.

**Por qué así:** el objetivo específico 1 pide *caracterizar* los eventos, lo que en metodología corresponde a un análisis **descriptivo** (frecuencias, sumas, distribución temporal y espacial; Hernández-Sampieri et al., 2014). No se usan pruebas estadísticas aquí porque no se busca contrastar una hipótesis, sino describir la amenaza para elegir los casos de los retos. La estacionalidad confirma lo que plantea la literatura (lluvias frontales de invierno; Garreaud, 2013), y el ranking de comunas justifica la elección de la comuna piloto.

**Cómo ampliarlo:** mapas por comuna (con los shapefiles que vienen en el mismo ZIP), tasas por población (fichas por cada 100 000 habitantes, con datos del Censo) o análisis de tendencia en el tiempo (prueba de Mann-Kendall).

In [ ]:
IMPACTOS = {"muertos":"Fallecidos","heridos":"Heridos","desaparece":"Desaparecidos","vivdest":"Viviendas destruidas",
            "vivafec":"Viviendas dañadas","damnificados":"Damnificados","evacuados":"Evacuados"}
IMPACTOS = {k: v for k, v in IMPACTOS.items() if k in rm}

orden = ["Aluvión","Avenida torrencial","Deslizamiento","Inundación","Lluvias","Temporal"]
t_tipo = rm.groupby("evento_norm").agg(Fichas=("evento_norm","size"), **{v: (k, "sum") for k, v in IMPACTOS.items()})
t_tipo = t_tipo.reindex([o for o in orden if o in t_tipo.index])
t_tipo.loc["Total"] = t_tipo.sum()
t_tipo = t_tipo.fillna(0).astype(int).reset_index().rename(columns={"evento_norm": "Tipo de evento"})
registrar_tabla(t_tipo, f"Fichas e impactos de eventos hidrometeorológicos en la Región Metropolitana, {ANIO_MIN}–{ANIO_MAX}",
                "Elaborado a partir de DesInventar Chile (LA RED, s.f.). Los impactos corresponden a la suma de las fichas.", "4.1")

In [ ]:
# Estacionalidad (mes) y evolución por década
meses = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
por_mes = rm[rm["fechames"].between(1,12)].groupby("fechames").size().reindex(range(1,13), fill_value=0)
rm["decada"] = (rm["fechano"]//10*10).astype(int).astype(str) + "s"
por_decada = rm.pivot_table(index="decada", columns="evento_norm", values="evento", aggfunc="size", fill_value=0)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(meses, por_mes.values, color="#4a4a4a"); ax[0].set_title("Fichas por mes"); ax[0].set_ylabel("N.º de fichas")
por_decada.plot(kind="bar", stacked=True, ax=ax[1], colormap="Greys", edgecolor="black", linewidth=.3)
ax[1].set_title("Fichas por década y tipo"); ax[1].set_xlabel(""); ax[1].legend(fontsize=7, frameon=False)
for a in ax: a.spines[["top","right"]].set_visible(False)
plt.tight_layout()
registrar_figura(fig, "fig_estacionalidad_decadas.png",
                 "Distribución mensual y por década de los eventos hidrometeorológicos en la Región Metropolitana",
                 "Elaborado a partir de DesInventar Chile (LA RED, s.f.).", "4.1")

pct_invierno = por_mes.loc[5:8].sum() / por_mes.sum() * 100 if por_mes.sum() else float("nan")
print(f"% de fichas entre mayo y agosto: {pct_invierno:.1f}%")

In [ ]:
# Comunas con más fichas
t_comunas = (rm.assign(Comuna=rm["name2"].replace("", "Sin comuna"))
               .groupby("Comuna").agg(Fichas=("evento","size"), Aluviones=("evento_norm", lambda s: (s=="Aluvión").sum()),
                                      **({"Viviendas dañadas": ("vivafec","sum")} if "vivafec" in rm else {}))
               .sort_values("Fichas", ascending=False).head(10).reset_index())
registrar_tabla(t_comunas, "Comunas de la Región Metropolitana con mayor número de fichas hidrometeorológicas",
                "Elaborado a partir de DesInventar Chile (LA RED, s.f.). Se muestran las 10 comunas con más fichas.", "4.1")

## 4. Datos hidrometeorológicos CAMELS-CL

Enlaces (versión 2018 publicada en PANGAEA, licencia CC-BY 4.0):
- Ficha del dataset: https://doi.pangaea.de/10.1594/PANGAEA.894885
- Atributos: https://store.pangaea.de/Publications/Alvarez-Garreton-etal_2018/1_CAMELScl_attributes.zip
- Precipitación CR2MET: https://store.pangaea.de/Publications/Alvarez-Garreton-etal_2018/4_CAMELScl_precip_cr2met.zip
- Temperatura media CR2MET: https://store.pangaea.de/Publications/Alvarez-Garreton-etal_2018/10_CAMELScl_tmean_cr2met.zip
- Visualizador: https://camels.cr2.cl

Cada serie es la media diaria de la cuenca (mm/día o °C). Se usan las cuencas cuya estación de aforo está dentro de la RM.

### 📘 Explicación para la defensa — CAMELS-CL
**Qué hace:** descarga tres archivos de CAMELS-CL (atributos de cuencas, precipitación diaria y temperatura media diaria), los lee y selecciona las cuencas cuya estación de aforo está dentro de la RM.

**Qué entrega:** tabla de cuencas utilizadas (sección 3.3) y las series diarias de precipitación y temperatura.

**Por qué CAMELS-CL y no otra fuente:**
- Entrega **series diarias continuas, sin vacíos y ya promediadas por cuenca** (producto CR2MET), listas para usar. Las estaciones de la Dirección Meteorológica de Chile (DMC) o de la DGA tienen vacíos, distintos formatos y requieren registro o solicitud.
- Es un dataset **publicado en una revista indexada** (Alvarez-Garreton et al., 2018) con licencia abierta CC-BY, lo que da trazabilidad académica.
- Promediar varias cuencas de la RM representa mejor la lluvia en la precordillera (donde se originan los aluviones) que una sola estación en el valle, como Quinta Normal.

**Limitación:** la versión 2018 cubre aproximadamente 1979–2016, por lo que el modelo usa 1979–2014 (la intersección con DesInventar).

**Cómo ampliarlo:** usar la versión 2022 de CAMELS-CL (más años), agregar la **isoterma 0 °C** (radiosonda de Santo Domingo, variable clave según Garreaud, 2013) o la cobertura de nieve.

In [ ]:
BASE_PANGAEA = "https://store.pangaea.de/Publications/Alvarez-Garreton-etal_2018/"
ARCH = {"attr": "1_CAMELScl_attributes", "precip": "4_CAMELScl_precip_cr2met", "tmean": "10_CAMELScl_tmean_cr2met"}
rutas_camels = {}
for k, nombre in ARCH.items():
    destino = os.path.join(DATA_DIR, nombre + ".zip")
    ruta = descargar([BASE_PANGAEA + nombre + ".zip"], destino, min_bytes=1_000)
    if ruta is None:
        ruta = pedir_archivo(f"Sube {nombre}.zip (enlace en la celda anterior):")
    rutas_camels[k] = ruta

def leer_txt_zip(ruta):
    if zipfile.is_zipfile(ruta):
        z = zipfile.ZipFile(ruta)
        n = [x for x in z.namelist() if x.lower().endswith((".txt", ".csv"))][0]
        return z.read(n).decode("utf-8", errors="replace")
    return open(ruta, encoding="utf-8", errors="replace").read()

def _limpiar(s): return str(s).strip().strip('"').strip()

def leer_atributos(ruta):
    txt = leer_txt_zip(ruta)
    df = pd.read_csv(io.StringIO(txt), sep="\t", dtype=str, header=None).map(_limpiar)
    # Formato PANGAEA: filas = atributos, columnas = cuencas (primera fila 'gauge_id')
    if df.iloc[0, 0].lower() == "gauge_id":
        df = df.set_index(0).T
    else:
        df.columns = df.iloc[0]; df = df.iloc[1:]
    df = df.rename(columns=str.strip).set_index("gauge_id")
    for c in ["gauge_lat", "gauge_lon", "area"]:
        if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def leer_serie(ruta):
    txt = leer_txt_zip(ruta)
    df = pd.read_csv(io.StringIO(txt), sep="\t", dtype=str, header=None).map(_limpiar)
    fechas_col = pd.to_datetime(df.iloc[:, 0], errors="coerce", format="mixed")
    # Variante: columnas separadas año / mes / día
    num = df.iloc[:, :3].apply(pd.to_numeric, errors="coerce")
    if fechas_col.notna().sum() < len(df) * 0.5 and num.iloc[:, 0].between(1900, 2100).mean() > 0.5:
        f = pd.to_datetime(dict(year=num.iloc[:, 0], month=num.iloc[:, 1], day=num.iloc[:, 2]), errors="coerce")
        cab = df.iloc[0].tolist(); datos = df[f.notna()].iloc[:, 3:]
        datos.columns = cab[3:]; datos.index = f[f.notna()]
        return datos.apply(pd.to_numeric, errors="coerce").pipe(lambda d: d.mask(d <= -999))
    if fechas_col.notna().sum() > len(df) * 0.5:          # filas = días
        cab = df.iloc[0].tolist(); datos = df[fechas_col.notna()]
        datos.columns = ["fecha"] + cab[1:]
    else:                                                  # columnas = días (formato transpuesto)
        df = df.T; fechas_col = pd.to_datetime(df.iloc[:, 0], errors="coerce", format="mixed")
        cab = df.iloc[0].tolist(); datos = df[fechas_col.notna()]
        datos.columns = ["fecha"] + cab[1:]
    datos = datos.set_index(pd.to_datetime(datos["fecha"])).drop(columns="fecha")
    datos = datos.apply(pd.to_numeric, errors="coerce")
    datos = datos.mask(datos <= -999)                      # valores faltantes
    return datos

attrs = leer_atributos(rutas_camels["attr"])
precip = leer_serie(rutas_camels["precip"])
tmean = leer_serie(rutas_camels["tmean"])
print("Cuencas:", len(attrs), "| precipitación:", precip.index.min().date(), "→", precip.index.max().date())

sel = attrs[attrs["gauge_lat"].between(*RM_LAT) & attrs["gauge_lon"].between(*RM_LON)]
cuencas_rm = [g for g in sel.index if g in precip.columns]
print("Cuencas seleccionadas en la RM:", len(cuencas_rm))
t_cuencas = sel.loc[cuencas_rm, [c for c in ["gauge_name","gauge_lat","gauge_lon","area"] if c in sel]].reset_index()
t_cuencas.columns = ["Código","Estación","Latitud","Longitud","Área (km²)"][:t_cuencas.shape[1]]
registrar_tabla(t_cuencas, "Cuencas CAMELS-CL utilizadas para caracterizar la precipitación en la Región Metropolitana",
                "Elaborado a partir de CAMELS-CL (Alvarez-Garreton et al., 2018).", "3.3")

## 5. Construcción de la base diaria (variables del modelo)

Para cada día se calcula, sobre el promedio de las cuencas de la RM:
- `P_dia`: precipitación del día (mm) — factor principal según Sepúlveda y Padilla (2008).
- `P_3d`, `P_7d`: precipitación acumulada de 3 y 7 días previos (mm).
- `T_media`: temperatura media (°C) — aproxima las tormentas cálidas con isoterma alta (Garreaud, 2013).
- `mes` (seno/coseno) para la estacionalidad.

Variable objetivo `evento = 1` si DesInventar registra al menos una ficha hidrometeorológica en la RM ese día (o durante su duración, hasta 3 días).

### 📘 Explicación para la defensa — Base diaria y variables del modelo
**Qué hace:** une ambas fuentes en una tabla con **una fila por día**. Para cada día calcula la lluvia del día, la lluvia acumulada de los 3 y 7 días previos, la temperatura media y la estacionalidad (mes convertido a seno y coseno). Marca `evento = 1` si ese día (o durante la duración reportada, hasta 3 días) DesInventar registra una ficha en la RM.

**Qué entrega:** la base de entrenamiento, una tabla que compara la lluvia en días con y sin evento, y un diagrama de cajas.

**Por qué estas variables:**
- **Lluvia del día y acumulada:** Sepúlveda y Padilla (2008) demostraron, para el piedemonte de Santiago, que la lluvia del día del evento es el factor desencadenante más importante, seguida de la acumulada.
- **Temperatura media:** aproxima las "tormentas cálidas", en que la isoterma alta hace que llueva en vez de nevar en la cordillera (Garreaud, 2013).
- **Mes como seno/coseno:** así diciembre y enero quedan "cerca", algo que el número del mes (12 y 1) no representa.
- Las acumuladas usan solo **días previos** (`shift`) para no "filtrar" información del mismo día dos veces.

**Por qué un día como unidad de análisis:** es la resolución de ambas fuentes y la escala a la que se emiten las alertas meteorológicas.

In [ ]:
P = precip[cuencas_rm].mean(axis=1).rename("P_dia")
T = tmean[[c for c in cuencas_rm if c in tmean.columns]].mean(axis=1).rename("T_media")
base = pd.concat([P, T], axis=1).dropna()
base["P_3d"] = base["P_dia"].rolling(3, min_periods=1).sum().shift(1).fillna(0)
base["P_7d"] = base["P_dia"].rolling(7, min_periods=1).sum().shift(1).fillna(0)
base["mes_sin"] = np.sin(2*np.pi*base.index.month/12); base["mes_cos"] = np.cos(2*np.pi*base.index.month/12)

dias_evento = set()
for _, r in rm[rm["fecha_valida"]].iterrows():
    dur = int(min(max(r.get("duracion", 0) if pd.notna(r.get("duracion", 0)) else 0, 0), 3))
    for k in range(dur + 1):
        dias_evento.add(r["fecha"].normalize() + pd.Timedelta(days=k))
base["evento"] = base.index.normalize().isin(dias_evento).astype(int)
base = base[(base.index.year >= max(ANIO_MIN, base.index.year.min())) & (base.index.year <= ANIO_MAX)]
print("Días en la base:", len(base), "| días con evento:", base["evento"].sum(),
      f"({base['evento'].mean()*100:.2f} %)", "| período:", base.index.min().date(), "→", base.index.max().date())

# Comparación de la lluvia en días con y sin evento
lluviosos = base[base["P_dia"] >= 1]
t_umbral = (base.groupby("evento")[["P_dia","P_3d","T_media"]]
              .describe(percentiles=[.5,.9]).loc[:, (slice(None), ["mean","50%","90%"])].round(1))
t_umbral.index = ["Días sin evento","Días con evento"]
t_umbral.columns = [f"{a} – { {'mean':'media','50%':'mediana','90%':'p90'}[b]}" for a, b in t_umbral.columns]
registrar_tabla(t_umbral.reset_index().rename(columns={"index":"Grupo"}),
                "Precipitación y temperatura en días con y sin eventos registrados en la Región Metropolitana",
                "P_dia = precipitación diaria (mm); P_3d = acumulada de los 3 días previos (mm); T_media en °C. Elaborado a partir de DesInventar Chile y CAMELS-CL.", "4.1")

fig, ax = plt.subplots(figsize=(6, 3.4))
datos_box = [base.loc[base.evento==0, "P_dia"].clip(upper=120), base.loc[base.evento==1, "P_dia"].clip(upper=120)]
ax.boxplot(datos_box, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor="#d9d9d9"), medianprops=dict(color="black"))
ax.set_xticks([1, 2]); ax.set_xticklabels(["Sin evento", "Con evento"])
ax.set_ylabel("Precipitación diaria (mm)"); ax.spines[["top","right"]].set_visible(False)
registrar_figura(fig, "fig_precip_eventos.png", "Precipitación diaria en días con y sin evento registrado",
                 "Valores atípicos omitidos. Elaborado a partir de DesInventar Chile y CAMELS-CL.", "4.1")

## 6. Entrenamiento y evaluación del modelo de IA → Tesis 4.2 (Objetivo específico 2)

Se comparan dos clasificadores, con pesos de clase balanceados (los días con evento son pocos):
1. **Regresión logística** (mismo enfoque de Sepúlveda y Padilla, 2008; interpretable).
2. **Bosque aleatorio** (no lineal).

Partición **temporal**: entrenamiento hasta `ANIO_FIN_ENTRENAMIENTO`, prueba en los años siguientes (evita que el modelo “vea el futuro”).

### 📘 Explicación para la defensa — Modelo de IA (Objetivo específico 2)
**Qué hace:** entrena dos modelos que estiman la **probabilidad de que ocurra un evento** dado el clima del día, los evalúa en años que no vio y traduce esa probabilidad a niveles de alerta Verde / Amarilla / Roja.

**Qué entrega:**
- **Tabla de desempeño** con estas métricas:
  - **ROC-AUC:** capacidad de ordenar días riesgosos sobre días normales (0,5 = azar; 1 = perfecto).
  - **PR-AUC:** la métrica más honesta cuando los eventos son raros. Se compara con la tasa base, que es lo que obtendría el azar.
  - **Sensibilidad:** porcentaje de eventos reales que el modelo detectó.
  - **Precisión:** de las alertas emitidas, cuántas fueron eventos reales.
  - **F1:** equilibrio entre sensibilidad y precisión.
  - **Brier:** error de las probabilidades.
  - **Matriz de confusión:** VP, FP, FN y VN.
- **Tabla de contribución de variables** (odds ratios e importancias) y la **curva ROC**.
- **Tabla de niveles de alerta**: qué porcentaje de eventos cae en cada color.

**Por qué estos modelos y no otros:**
- **Regresión logística:** es el mismo método usado en la literatura local (Sepúlveda y Padilla, 2008), lo que permite comparar. Además es **interpretable**: cada coeficiente se traduce en "por cada X mm adicionales, la probabilidad se multiplica por Y". Esto es coherente con la tesis, que trata sobre **interpretar** salidas de IA (Barredo Arrieta et al., 2020).
- **Bosque aleatorio:** captura relaciones no lineales (por ejemplo, que la lluvia sea peligrosa solo si además hace calor) y es robusto con pocos datos; sirve para ver si un modelo más complejo mejora en algo.
- **¿Por qué no redes neuronales o LSTM** (como Nearing et al., 2024)? Con unos pocos cientos de días con evento, una red profunda se **sobreajusta** y no se puede explicar. El alcance de la tesis (1.7) aclara que el modelo debe generar salidas realistas para la plataforma, no competir en precisión.

**Por qué se valida así:**
- **Partición temporal** (entrenar hasta 2006 y probar 2007–2014) en vez de una partición aleatoria: simula el uso real (predecir el futuro con el pasado) y evita que días consecutivos de un mismo temporal queden repartidos entre entrenamiento y prueba, lo que inflaría artificialmente el desempeño.
- **Pesos balanceados por clase:** los días con evento son pocos (alrededor del 1–3 %). Sin balanceo, el modelo "aprende" a decir siempre "no hay evento" y obtiene un 97 % de exactitud sin servir para nada. Por eso **no se reporta la exactitud (accuracy)**.
- **Umbral elegido en entrenamiento:** no se ajusta mirando los datos de prueba, porque eso sería hacer trampa.
- **Niveles de alerta por percentiles (95 y 99):** dan una frecuencia de alertas realista (amarilla unos 15 días al año y roja unos 4). En un sistema real, los umbrales se calibrarían con Senapred.

**Cómo ampliarlo:**
- **Más modelos:** XGBoost o LightGBM, o un modelo por comuna.
- **Más variables:** isoterma 0 °C, pendiente y área quemada de las quebradas (el aluvión de 2026 ocurrió tras un incendio) o nieve acumulada.
- **Validación cruzada temporal** (`TimeSeriesSplit`) y **calibración de probabilidades**.
- **Explicaciones por predicción con SHAP**, que la plataforma podría mostrar al usuario: "la alerta es roja porque llovieron 45 mm".

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score, recall_score, precision_score,
                             f1_score, confusion_matrix, roc_curve, brier_score_loss)

X_COLS = ["P_dia","P_3d","P_7d","T_media","mes_sin","mes_cos"]
train = base[base.index.year <= ANIO_FIN_ENTRENAMIENTO]; test = base[base.index.year > ANIO_FIN_ENTRENAMIENTO]
Xtr, ytr, Xte, yte = train[X_COLS], train["evento"], test[X_COLS], test["evento"]
print(f"Entrenamiento: {len(train)} días ({ytr.sum()} con evento) | Prueba: {len(test)} días ({yte.sum()} con evento)")

modelos = {
    "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000)),
    "Bosque aleatorio": RandomForestClassifier(n_estimators=400, min_samples_leaf=20, class_weight="balanced_subsample",
                                               random_state=SEMILLA, n_jobs=-1),
}
filas, probas = [], {}
for nombre, m in modelos.items():
    m.fit(Xtr, ytr)
    p = m.predict_proba(Xte)[:, 1]; probas[nombre] = p
    # umbral elegido en entrenamiento: el que maximiza F1
    ptr = m.predict_proba(Xtr)[:, 1]
    umbrales = np.quantile(ptr, np.linspace(0.80, 0.999, 200))
    u = max(umbrales, key=lambda t: f1_score(ytr, ptr >= t, zero_division=0))
    yhat = (p >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, yhat, labels=[0,1]).ravel()
    filas.append({"Modelo": nombre, "ROC-AUC": roc_auc_score(yte, p) if yte.nunique() > 1 else np.nan,
                  "PR-AUC": average_precision_score(yte, p) if yte.sum() else np.nan,
                  "Sensibilidad (recall)": recall_score(yte, yhat, zero_division=0),
                  "Precisión": precision_score(yte, yhat, zero_division=0), "F1": f1_score(yte, yhat, zero_division=0),
                  "Brier": brier_score_loss(yte, p), "VP": tp, "FP": fp, "FN": fn, "VN": tn})
t_modelos = pd.DataFrame(filas).round(3)
t_modelos.loc[len(t_modelos)] = {"Modelo": "Referencia: azar (tasa base)", "ROC-AUC": 0.5, "PR-AUC": round(yte.mean(), 3)}
registrar_tabla(t_modelos, f"Desempeño de los modelos en el período de prueba ({ANIO_FIN_ENTRENAMIENTO+1}–{test.index.year.max()})",
                "VP = verdaderos positivos; FP = falsos positivos; FN = falsos negativos; VN = verdaderos negativos. "
                "El umbral de clasificación se fijó en el período de entrenamiento maximizando F1.", "4.2")

In [ ]:
# Interpretabilidad: razones de probabilidad (odds ratios) de la regresión logística e importancia del bosque
lr = modelos["Regresión logística"]; coefs = lr[-1].coef_[0]
t_or = pd.DataFrame({"Variable": X_COLS, "Coeficiente (estandarizado)": coefs.round(3),
                     "Odds ratio por 1 DE": np.exp(coefs).round(2),
                     "Importancia (bosque aleatorio)": modelos["Bosque aleatorio"].feature_importances_.round(3)})
t_or = t_or.sort_values("Importancia (bosque aleatorio)", ascending=False)
registrar_tabla(t_or, "Contribución de cada variable a la predicción de eventos",
                "DE = desviación estándar. Un odds ratio mayor que 1 indica que la variable aumenta la probabilidad de evento.", "4.2")

fig, ax = plt.subplots(figsize=(4.6, 4.2))
for (nombre, p), est in zip(probas.items(), ["-", "--"]):
    if yte.nunique() > 1:
        fpr, tpr, _ = roc_curve(yte, p); ax.plot(fpr, tpr, est, color="black", label=f"{nombre} (AUC = {roc_auc_score(yte, p):.2f})")
ax.plot([0,1],[0,1], ":", color="grey", label="Azar"); ax.set_xlabel("Tasa de falsos positivos"); ax.set_ylabel("Sensibilidad")
ax.legend(fontsize=8, frameon=False); ax.spines[["top","right"]].set_visible(False)
registrar_figura(fig, "fig_roc.png", "Curvas ROC de los modelos en el período de prueba",
                 "Elaborado a partir de DesInventar Chile y CAMELS-CL.", "4.2")

In [ ]:
# Niveles de alerta para la plataforma: se traducen probabilidades del modelo a Verde / Amarilla / Roja
mejor = t_modelos.iloc[:2].sort_values("PR-AUC", ascending=False)["Modelo"].iloc[0]
m = modelos[mejor]; p_all = m.predict_proba(base[X_COLS])[:, 1]
u_amar, u_roja = np.quantile(p_all, 0.95), np.quantile(p_all, 0.99)
base["prob"] = p_all
base["alerta"] = np.select([p_all >= u_roja, p_all >= u_amar], ["Roja", "Amarilla"], "Verde")
t_alertas = (base.groupby("alerta").agg(Días=("evento","size"), **{"Días con evento": ("evento","sum")})
             .reindex(["Verde","Amarilla","Roja"]))
t_alertas["% con evento"] = (t_alertas["Días con evento"] / t_alertas["Días"] * 100).round(2)
t_alertas["% de los eventos capturados"] = (t_alertas["Días con evento"] / base["evento"].sum() * 100).round(1)
registrar_tabla(t_alertas.reset_index().rename(columns={"alerta":"Nivel de alerta"}),
                f"Niveles de alerta generados por el modelo seleccionado ({mejor})",
                f"Amarilla: probabilidad ≥ percentil 95 ({u_amar:.3f}); Roja: ≥ percentil 99 ({u_roja:.3f}). Todo el período analizado.", "4.2")

## 7. Selección de casos históricos para los retos → Tesis 3.5 (paso 3) y 4.2
Se eligen los eventos con mayor impacto que tienen datos hidrometeorológicos, priorizando aluviones.

### 📘 Explicación para la defensa — Selección de casos para los retos
**Qué hace:** agrupa las fichas por fecha, les une la lluvia, la temperatura y la alerta que habría dado el modelo, y ordena los eventos según un **puntaje de impacto**: fallecidos ×10, viviendas destruidas ×2, viviendas dañadas ×0,1, damnificados ×0,01, y un bono de 50 si incluye aluvión o avenida.

**Qué entrega:** los 8 eventos candidatos para construir los 3–5 escenarios de la plataforma (paso 3 del procedimiento).

**Por qué así:** el ABR exige **retos auténticos** (Gallagher y Savage, 2023). Elegir eventos reales y de alto impacto, con sus datos de lluvia, hace que el usuario enfrente la misma información que tuvieron los equipos de emergencia. Los pesos del puntaje son un **criterio explícito y trazable** (no una elección "a dedo"), aunque son una decisión de diseño. Si les preguntan, pueden decir que se pueden ajustar, o validar la lista con el experto en gestión del riesgo que revisará los instrumentos.

**Cómo ampliarlo:** que el experto priorice la lista; incluir casos donde el modelo **falló** (alerta verde con evento real), que son muy buenos retos para enseñar los límites de la IA; agregar eventos posteriores a 2014 desde reportes de Sernageomin y Senapred.

In [ ]:
cand = rm[rm["fecha_valida"]].copy()
cand["fecha_d"] = cand["fecha"].dt.normalize()
agg = {"Tipos": ("evento_norm", lambda s: ", ".join(sorted(set(s)))), "Comunas": ("name2", lambda s: ", ".join(sorted(set(x for x in s if x)))[:80]),
       "Fichas": ("evento","size")}
for k, v in IMPACTOS.items(): agg[v] = (k, "sum")
casos = cand.groupby("fecha_d").agg(**agg)
casos = casos.join(base[["P_dia","P_3d","T_media","prob","alerta"]], how="inner")
casos.index.name = "fecha_d"   # pandas 2.2 (Colab) pierde el nombre del índice al unir
if casos.empty:
    raise ValueError("Ningún evento de la RM coincide con fechas de CAMELS-CL. Revisa que 'fecha_valida' tenga registros (sección 2) "
                     "y que el período de precipitación cubra 1979–2014 (sección 4).")
casos["puntaje"] = casos.get("Fallecidos", 0)*10 + casos.get("Viviendas destruidas", 0)*2 + casos.get("Viviendas dañadas", 0)*0.1 \
                   + casos.get("Damnificados", 0)*0.01 + casos["Tipos"].str.contains("Aluvión|Avenida")*50
casos = casos.sort_values("puntaje", ascending=False).head(8).round(2)
t_casos = casos.drop(columns="puntaje").reset_index().rename(columns={"fecha_d":"Fecha","P_dia":"P día (mm)","P_3d":"P 3 días (mm)",
                                                                       "T_media":"T media (°C)","prob":"Prob. modelo","alerta":"Alerta modelo"})
t_casos["Fecha"] = pd.to_datetime(t_casos["Fecha"]).dt.strftime("%d-%m-%Y")
t_casos["Prob. modelo"] = t_casos["Prob. modelo"].round(3)
registrar_tabla(t_casos, "Eventos históricos candidatos para construir los escenarios de los retos",
                "Ordenados por un puntaje de impacto que prioriza fallecidos, viviendas y aluviones. Elaborado a partir de DesInventar Chile y CAMELS-CL.", "4.2")

## 8. Análisis del piloto (pre–post) → Tesis 4.3 (Objetivo específico 4) y prueba de hipótesis

**Esta sección solo funciona con los datos reales del piloto.** La celda crea `plantilla_piloto.csv` con estas columnas:

| Columna | Descripción |
|---|---|
| `id` | Código anónimo del participante (P01, P02…) |
| `grupo` | `estudiante` o `funcionario` |
| `comprension_pre`, `comprension_post` | Puntaje del cuestionario de comprensión (0–100) |
| `decision_pre`, `decision_post` | Puntaje de la rúbrica de calidad de la decisión (0–100) |
| `usabilidad` | Puntaje del cuestionario de usabilidad (p. ej. SUS, 0–100) |

Llénenla (en Excel o Google Sheets), guárdenla como CSV y súbanla cuando se pida. Se aplica prueba *t* pareada o Wilcoxon según la normalidad de las diferencias (Shapiro-Wilk), con α = 0,05 y tamaño del efecto.

### 📘 Explicación para la defensa — Análisis del piloto (Objetivo específico 4 e hipótesis)
**Qué hace:** crea la plantilla para registrar los puntajes de cada participante antes y después de usar la plataforma. Cuando hay datos, compara ambos momentos:
1. **Shapiro-Wilk** sobre las diferencias (post − pre) para ver si son normales.
2. Si son normales, aplica la **prueba t para muestras relacionadas**; si no, la **prueba de Wilcoxon**, que es no paramétrica.
3. Calcula el **tamaño del efecto**: *d* de Cohen (dz) o *r*. Como referencia, 0,2 es un efecto pequeño, 0,5 mediano y 0,8 grande.
4. Decide sobre H0 con α = 0,05.

**Qué entrega:** la tabla pre–post para la sección 4.3 y la usabilidad por grupo.

**Por qué así:** el diseño es preexperimental de **un grupo con medición antes y después** (sección 3.1). Las mismas personas se miden dos veces, por lo que corresponde una prueba **pareada**, no una para grupos independientes. Con muestras pequeñas (15–25 personas) la normalidad no está garantizada, por eso se verifica y se usa Wilcoxon como alternativa. El tamaño del efecto se reporta porque, con pocos participantes, un valor p no significativo no significa necesariamente que no haya efecto.

**Cómo usarlo:** llenen la plantilla en Excel (sirve con punto y coma y decimales con coma), guárdenla en `datos/` con la palabra *piloto* en el nombre, o cambien `SUBIR_PILOTO_AHORA = True`.

**Cómo ampliarlo:** comparar estudiantes y funcionarios (Mann-Whitney), correlacionar el tiempo en cada fase ABR con la mejora, analizar las respuestas abiertas o, en un estudio futuro, agregar un **grupo control** (diseño cuasiexperimental).

In [ ]:
from scipy import stats
cols_piloto = ["id","grupo","comprension_pre","comprension_post","decision_pre","decision_post","usabilidad"]
pd.DataFrame(columns=cols_piloto).to_csv(os.path.join(OUT_DIR, "plantilla_piloto.csv"), index=False)
print("Plantilla creada en", os.path.join(OUT_DIR, "plantilla_piloto.csv"))

# Para analizar el piloto: sube el CSV/XLSX a la carpeta 'datos/' (con 'piloto' en el nombre) o cambia esta variable a True
SUBIR_PILOTO_AHORA = False
ruta_piloto = next(iter(sorted(glob.glob(os.path.join(DATA_DIR, "*piloto*.*")))), None)
if ruta_piloto is None and EN_COLAB and SUBIR_PILOTO_AHORA:
    ruta_piloto = pedir_archivo("Sube el CSV o XLSX del piloto:")

def analizar_par(pre, post, nombre):
    d = (post - pre).dropna(); n = len(d)
    sw_p = stats.shapiro(d).pvalue if n >= 3 else np.nan
    if n >= 3 and sw_p > 0.05:
        t = stats.ttest_rel(post, pre, nan_policy="omit"); prueba, est, p = "t pareada", t.statistic, t.pvalue
        efecto, nombre_ef = d.mean() / d.std(ddof=1), "d de Cohen (dz)"
    else:
        if (d == 0).all():
            return {"Variable": nombre, "n": n, "Prueba": "—", "Decisión (α = 0,05)": "Sin variación entre pre y post"}
        w = stats.wilcoxon(post, pre, nan_policy="omit"); prueba, est, p = "Wilcoxon", w.statistic, w.pvalue
        z = stats.norm.isf(p / 2) * np.sign(d.median() if d.median() != 0 else 1); efecto, nombre_ef = z / np.sqrt(n), "r"
    return {"Variable": nombre, "n": n, "Pre M (DE)": f"{pre.mean():.1f} ({pre.std():.1f})", "Post M (DE)": f"{post.mean():.1f} ({post.std():.1f})",
            "Diferencia media": round(d.mean(), 2), "Shapiro-Wilk p": round(sw_p, 3), "Prueba": prueba,
            "Estadístico": round(est, 3), "p": round(p, 4), "Tamaño del efecto": f"{nombre_ef} = {efecto:.2f}",
            "Decisión (α = 0,05)": "Se rechaza H0" if p < 0.05 else "No se rechaza H0"}

def leer_piloto(ruta):
    # Acepta CSV con coma o punto y coma (Excel en español) y decimales con coma; también .xlsx
    if ruta.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(ruta)
    else:
        df = pd.read_csv(ruta, sep=None, engine="python", encoding_errors="replace")
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    faltan = [c for c in cols_piloto if c not in df.columns]
    if faltan:
        raise ValueError(f"Al CSV del piloto le faltan columnas: {faltan}. Usa la plantilla 'plantilla_piloto.csv'.")
    for c in cols_piloto[2:]:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", ".", regex=False).str.strip(), errors="coerce")
    return df

if ruta_piloto:
    pil = leer_piloto(ruta_piloto)
    pil = pil.dropna(subset=["comprension_pre","comprension_post"], how="all")
    if len(pil) < 3:
        print("⚠️ El CSV tiene menos de 3 participantes con datos: no se puede analizar todavía.")
    else:
        t_piloto = pd.DataFrame([analizar_par(pil["comprension_pre"], pil["comprension_post"], "Comprensión de salidas de IA"),
                                 analizar_par(pil["decision_pre"], pil["decision_post"], "Calidad de la decisión")])
        registrar_tabla(t_piloto, "Comparación pre–post de comprensión y calidad de la decisión en el grupo piloto",
                        "M = media; DE = desviación estándar. Prueba t pareada si las diferencias son normales (Shapiro-Wilk p > 0,05); en caso contrario, Wilcoxon.", "4.3")
        if "usabilidad" in pil and pil["usabilidad"].notna().any():
            t_us = pil.groupby("grupo")["usabilidad"].describe()[["count","mean","std","min","max"]].round(1).reset_index()
            t_us.columns = ["Grupo","n","Media","DE","Mín.","Máx."]
            registrar_tabla(t_us, "Percepción de usabilidad de la plataforma por grupo", "Escala 0–100.", "4.3")
else:
    print("Sin datos del piloto todavía: la sección 4.3 de la tesis queda pendiente hasta aplicar el piloto.")

## 9. Exportar tablas y figuras a Word (formato APA) + texto de apoyo

Genera `resultados/Resultados_tesis.docx` con todas las tablas (Tabla N en negrita, título en cursiva, *Nota.*) y figuras, más un borrador de redacción con las cifras. Las tablas se numeran desde `TABLA_INICIAL` porque la tesis ya tiene Tablas 1–2 en capítulos anteriores.

### 📘 Explicación para la defensa — Exportación
**Qué hace:** escribe todas las tablas y figuras registradas en un Word con formato APA 7 (número en negrita, título en cursiva, *Nota.*) y genera un borrador de redacción con las cifras reales. Además, descarga todo en un ZIP.

**Por qué así:** copiar números a mano desde la pantalla es la principal fuente de errores en una tesis. Generar las tablas directamente desde el código garantiza que lo que está en el documento sea **exactamente** lo que produjo el análisis, y si cambian un parámetro basta con volver a ejecutar.

In [ ]:
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

TABLA_INICIAL, FIGURA_INICIAL = 3, 2

def borde(celda, lado, sz=8):
    tcPr = celda._tc.get_or_add_tcPr(); b = tcPr.find(qn("w:tcBorders"))
    if b is None: b = OxmlElement("w:tcBorders"); tcPr.append(b)
    e = OxmlElement(f"w:{lado}"); e.set(qn("w:val"), "single"); e.set(qn("w:sz"), str(sz)); e.set(qn("w:color"), "000000"); b.append(e)

def fmt(v):
    if isinstance(v, (float, np.floating)):
        return "—" if pd.isna(v) else f"{v:,.3f}".rstrip("0").rstrip(".").replace(",", "X").replace(".", ",").replace("X", ".")
    return "" if pd.isna(v) else str(v)

doc = Document()
st = doc.styles["Normal"]; st.font.name = "Times New Roman"; st.font.size = Pt(12)
st.element.rPr.rFonts.set(qn("w:eastAsia"), "Times New Roman")
doc.add_heading("Resultados generados por el notebook", level=1)
doc.add_paragraph("Copie cada tabla/figura en la sección indicada de la tesis. Revise la numeración y el texto antes de pegar.")

for i, t in enumerate(TABLAS):
    n = TABLA_INICIAL + i; df = t["df"]
    doc.add_paragraph(f"[Sección {t['seccion']} de la tesis]").runs[0].italic = True
    p = doc.add_paragraph(); p.add_run(f"Tabla {n}").bold = True
    p = doc.add_paragraph(); p.add_run(t["titulo"]).italic = True
    tab = doc.add_table(rows=1, cols=len(df.columns))
    for j, c in enumerate(df.columns):
        cel = tab.rows[0].cells[j]; cel.text = str(c); borde(cel, "top"); borde(cel, "bottom", 6)
    for _, fila in df.iterrows():
        cs = tab.add_row().cells
        for j, v in enumerate(fila): cs[j].text = fmt(v)
    for cel in tab.rows[-1].cells: borde(cel, "bottom")
    for fila in tab.rows:
        for cel in fila.cells:
            for par in cel.paragraphs:
                for r in par.runs: r.font.size = Pt(9)
    p = doc.add_paragraph(); p.add_run("Nota. ").italic = True; p.add_run(t["nota"])
    doc.add_paragraph()

for i, f in enumerate(FIGURAS):
    n = FIGURA_INICIAL + i
    doc.add_paragraph(f"[Sección {f['seccion']} de la tesis]").runs[0].italic = True
    p = doc.add_paragraph(); p.add_run(f"Figura {n}").bold = True
    p = doc.add_paragraph(); p.add_run(f["titulo"]).italic = True
    doc.add_picture(f["ruta"], width=Inches(6.0))
    p = doc.add_paragraph(); p.add_run("Nota. ").italic = True; p.add_run(f["nota"])
    doc.add_paragraph()

# ---- Borrador de redacción con cifras reales ----
fila_tot = t_tipo[t_tipo["Tipo de evento"] == "Total"].iloc[0]
tipo_top = t_tipo[t_tipo["Tipo de evento"] != "Total"].sort_values("Fichas", ascending=False).iloc[0]
mejor_fila = t_modelos[t_modelos["Modelo"] == mejor].iloc[0]
def c(x, d=1): return f"{x:.{d}f}".replace(".", ",")
NOMBRES = {"P_dia": "la precipitación del día", "P_3d": "la precipitación acumulada de 3 días", "P_7d": "la precipitación acumulada de 7 días",
           "T_media": "la temperatura media", "mes_sin": "la estacionalidad", "mes_cos": "la estacionalidad"}
texto = [
    ("4.1", f"Entre {ANIO_MIN} y {ANIO_MAX}, DesInventar Chile registra {int(fila_tot['Fichas'])} fichas de eventos hidrometeorológicos "
            f"en la Región Metropolitana (Tabla {TABLA_INICIAL}). El tipo más frecuente es «{tipo_top['Tipo de evento']}» "
            f"({int(tipo_top['Fichas'])} fichas). El {c(pct_invierno)} % de las fichas se concentra entre mayo y agosto (Figura {FIGURA_INICIAL}), "
            f"lo que confirma el carácter invernal de la amenaza. Los días con evento presentan una precipitación media de "
            f"{c(base.loc[base.evento==1,'P_dia'].mean())} mm, frente a {c(base.loc[base.evento==0,'P_dia'].mean())} mm en los días sin evento."),
    ("4.2", f"En el período de prueba ({ANIO_FIN_ENTRENAMIENTO+1}–{test.index.year.max()}), el modelo con mejor desempeño fue «{mejor}», "
            f"con un ROC-AUC de {c(mejor_fila['ROC-AUC'], 2)} y una sensibilidad de {c(mejor_fila['Sensibilidad (recall)'], 2)}. "
            f"La variable más influyente fue {NOMBRES.get(t_or.iloc[0]['Variable'], t_or.iloc[0]['Variable'])}, en línea con lo reportado por Sepúlveda y Padilla (2008). "
            f"El nivel de alerta roja capturó el {c(t_alertas.loc['Roja','% de los eventos capturados'])} % de los días con evento."),
]
doc.add_heading("Borrador de redacción (revisar antes de usar)", level=2)
for sec, tx in texto:
    doc.add_paragraph(f"[{sec}] " + tx)

ruta_doc = os.path.join(OUT_DIR, "Resultados_tesis.docx"); doc.save(ruta_doc)
for sec, tx in texto: print(f"\n[{sec}] {tx}")
print("\nArchivo generado:", ruta_doc)

import shutil
shutil.make_archive("resultados_tesis", "zip", OUT_DIR)
if EN_COLAB:
    colab_files.download("resultados_tesis.zip")

## 🎓 Preguntas probables en la defensa (y cómo responderlas)

**1. ¿Por qué no usaron deep learning si la tesis se llama "Entrenamiento de IA"?**
Porque con unos pocos cientos de días con evento una red profunda se sobreajusta y es una "caja negra". La regresión logística y el bosque aleatorio son aprendizaje automático (IA) y además interpretables, que es justamente lo que la tesis busca enseñar. El aporte no es la precisión del modelo, sino la formación en la interpretación de sus salidas (sección 1.7).

**2. Su modelo tiene muchos falsos positivos.**
Es esperable y deseable en gestión del riesgo: un falso negativo (no alertar un aluvión) cuesta vidas; un falso positivo cuesta una evacuación preventiva. Por eso se prioriza la sensibilidad y se reporta PR-AUC frente a la tasa base, no la exactitud.

**3. ¿Por qué incluyen "temporal" o "lluvias" si el tema son los aluviones?**
Porque en DesInventar muchos aluviones quedaron registrados bajo esas categorías, y con solo las fichas "aluvión" no hay datos suficientes para entrenar. La tabla de caracterización muestra el detalle por tipo, de modo que se puede ver cuánto aporta cada uno.

**4. ¿Qué tan confiables son los datos?**
DesInventar se basa principalmente en prensa (sesgo de cobertura, reconocido en las limitaciones, sección 5.1) y CAMELS-CL es un producto grillado validado (Alvarez-Garreton et al., 2018). Por eso los resultados se presentan como evidencia para construir escenarios realistas, no como un sistema de alerta operativo.

**5. ¿El piloto prueba que la plataforma funciona?**
Entrega evidencia inicial. Al no haber grupo control, la mejora podría deberse en parte a la práctica con el mismo cuestionario. Esto se reconoce como limitación y se propone un diseño cuasiexperimental como trabajo futuro.

**6. ¿Por qué esos umbrales de alerta?**
Percentiles 95 y 99 de la probabilidad: dan una frecuencia de alertas razonable para un ejercicio formativo. En la plataforma se presentan como alertas simuladas; en un sistema real se calibrarían con Senapred.

## 🚀 Posibilidades de expansión del proyecto
1. **Datos:** extender a 2015–2026 con reportes de Senapred y Sernageomin y con la base de Escenarios Hídricos 2030; incorporar la isoterma 0 °C y las áreas quemadas (relevante para el aluvión de Las Condes de 2026).
2. **Modelo:** XGBoost o LightGBM, validación cruzada temporal, calibración, modelos por comuna o por quebrada, y explicaciones SHAP visibles en la plataforma.
3. **Espacial:** mapas de riesgo por comuna con los shapefiles de DesInventar y la cartografía de quebradas de Sernageomin.
4. **Plataforma:** generar automáticamente nuevos escenarios con lluvias "sintéticas" (por ejemplo, el temporal de 1993 con 20 % más de lluvia) para retos de "qué pasaría si".
5. **Evaluación:** grupo control, medición de retención a las 4 semanas y aplicación en más de un municipio.
6. **Otras amenazas y regiones:** misma arquitectura para incendios forestales, marejadas u otras regiones (Atacama 2015, Valparaíso).

## Referencias de los datos
- Alvarez-Garreton, C., Mendoza, P. A., Boisier, J. P., Addor, N., Galleguillos, M., Zambrano-Bigiarini, M., Lara, A., Puelma, C., Cortes, G., Garreaud, R., McPhee, J., & Ayala, A. (2018). The CAMELS-CL dataset: Catchment attributes and meteorology for large sample studies – Chile dataset. *Hydrology and Earth System Sciences, 22*(11), 5817–5846. https://doi.org/10.5194/hess-22-5817-2018
- Red de Estudios Sociales en Prevención de Desastres en América Latina. (s.f.). *DesInventar Chile: Inventario histórico de desastres, 1970–2014* [Conjunto de datos]. UNDRR. https://www.desinventar.net
- Sepúlveda, S. A., & Padilla, C. (2008). Rain-induced debris and mudflow triggering factors assessment in the Santiago cordilleran foothills, Central Chile. *Natural Hazards, 47*(2), 201–215. https://doi.org/10.1007/s11069-007-9210-6